In [1]:
#connect  google  drive  with google colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ==============================================================
#           EMAIL SPAM DETECTION USING LOGISTIC REGRESSION
# ==============================================================
# Project Name : Email Spam Detection
# Algorithm    : Logistic Regression
# Dataset      : us_email_dataset_10000_rows.csv
# Author       : Reem Fayyaz
# ==============================================================

# ==============================================================
# Install Required Libraries
# ==============================================================

# Install scikit-learn and joblib (Required only in Google Colab)
!pip -q install scikit-learn joblib

# ==============================================================
# Import Required Libraries
# ==============================================================

# Pandas is used for data loading and preprocessing
import pandas as pd

# NumPy is used for numerical operations
import numpy as np

# Joblib is used to save trained machine learning models
import joblib

# Train-Test Split divides data into Training and Testing sets
from sklearn.model_selection import train_test_split

# TF-IDF converts text into numerical vectors
from sklearn.feature_extraction.text import TfidfVectorizer

# Logistic Regression Classification Algorithm
from sklearn.linear_model import LogisticRegression

# Evaluation Metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# ==============================================================
# Load Dataset
# ==============================================================

# Read Email Spam Dataset
df = pd.read_csv("/content/drive/MyDrive/Module #4/us_email_dataset_10000_rows.csv")

# Display dataset size
print("="*60)
print("Dataset Shape :",df.shape)
print("="*60)

# Display first five records
print(df.head())

# Display dataset information
print("\nDataset Information")
print(df.info())

# Check Missing Values
print("\nMissing Values")
print(df.isnull().sum())

# ==============================================================
# Data Cleaning
# ==============================================================

# Remove duplicate records
df.drop_duplicates(inplace=True)

# Replace missing Subject with empty string
df["subject"] = df["subject"].fillna("")

# Replace missing Email Body with empty string
df["email_body"] = df["email_body"].fillna("")

# Combine Subject and Email Body into one feature
df["text"] = df["subject"] + " " + df["email_body"]

# Convert Labels into Numeric Values
# Ham = 0
# Spam = 1
df["label"] = df["label"].map({
    "Ham":0,
    "Spam":1
})

# Display Label Distribution
print("\nLabel Distribution")
print(df["label"].value_counts())

# Store Input Feature
X = df["text"]

# Store Target Variable
y = df["label"]

# ==========================
# TEXT VECTORIZATION
# ==========================

# Convert text into TF-IDF numerical features
# stop_words removes common English words
# max_features keeps only top 5000 important words
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

# Learn vocabulary and convert text into vectors
X_vector = vectorizer.fit_transform(X)


# ==========================
# TRAIN TEST SPLIT
# ==========================

# Split data into 80% training and 20% testing
# random_state ensures same split every time
X_train, X_test, y_train, y_test = train_test_split(
    X_vector,
    y,
    test_size=0.20,
    random_state=42
)

Dataset Shape : (10000, 8)
   email_id      sender_name                  sender_email    recipient_email  \
0         1   Ashley Jackson     ashley.jackson1@yahoo.com    user1@gmail.com   
1         2      John Miller      john.miller2@hotmail.com    user2@yahoo.com   
2         3      John Thomas        john.thomas3@gmail.com  user3@hotmail.com   
3         4    Daniel Taylor      daniel.taylor4@gmail.com  user4@outlook.com   
4         5  William Jackson  william.jackson5@outlook.com    user5@yahoo.com   

                  subject                                         email_body  \
0          Lottery Winner  Congratulations! Click the secure link now to ...   
1          Meeting Update  Hello, this is regarding meeting update. Pleas...   
2     Verify Your Account  Congratulations! Click the secure link now to ...   
3       Dinner Invitation  Hello, this is regarding dinner invitation. Pl...   
4  Urgent Action Required  Congratulations! Click the secure link now to ...   

   ca

In [ ]:
#(Model Training + Evaluation + Save PKL + User Prediction)

# ==============================================================
#            LOGISTIC REGRESSION MODEL TRAINING
# ==============================================================

# Create Logistic Regression Model
# max_iter increases training iterations
# random_state ensures reproducible results
# solver='liblinear' works well for binary classification
model = LogisticRegression(
    C=5,
    solver="liblinear",
    max_iter=3000,
    random_state=42
)

# Train the Logistic Regression model using training data
model.fit(X_train, y_train)

print("\n" + "="*60)
print("✅ Logistic Regression Model Trained Successfully")
print("="*60)

# ==============================================================
#                 MODEL PREDICTION
# ==============================================================

# Predict labels for testing dataset
y_pred = model.predict(X_test)

# Predict probability for each class
y_prob = model.predict_proba(X_test)

# ==============================================================
#                MODEL EVALUATION
# ==============================================================

# Calculate model accuracy
accuracy = accuracy_score(y_test, y_pred)

print("\n" + "="*60)
print("MODEL PERFORMANCE")
print("="*60)

print(f"Accuracy : {accuracy*100:.2f}%")

# Print Precision, Recall and F1-Score
print("\nClassification Report")
print(classification_report(y_test, y_pred))

# Print Confusion Matrix
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

# ==============================================================
#                 SAVE TRAINED MODEL
# ==============================================================

# Save Logistic Regression model
joblib.dump(model, "spam_model.pkl")

# Save TF-IDF Vectorizer
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("\n" + "="*60)
print("MODEL FILES SAVED SUCCESSFULLY")
print("="*60)

print("✔ spam_model.pkl")
print("✔ tfidf_vectorizer.pkl")

# ==============================================================
#               LOAD MODEL FOR PREDICTION
# ==============================================================

# Load saved model
loaded_model = joblib.load("spam_model.pkl")

# Load saved vectorizer
loaded_vectorizer = joblib.load("tfidf_vectorizer.pkl")

# ==============================================================
#                USER EMAIL SPAM CHECKER
# ==============================================================

print("\n" + "="*60)
print("📧 EMAIL SPAM DETECTION SYSTEM")
print("Type 'exit' to stop the program.")
print("="*60)

while True:

    # Take email text from user
    email = input("\nEnter Email Text : ")

    # Exit condition
    if email.lower() == "exit":
        print("\nProgram Closed Successfully.")
        break

    # Convert email into TF-IDF vector
    email_vector = loaded_vectorizer.transform([email])

    # Predict Spam or Ham
    prediction = loaded_model.predict(email_vector)[0]

    # Calculate probability
    probability = loaded_model.predict_proba(email_vector)[0]

    # Probability of Ham
    ham_probability = probability[0] * 100

    # Probability of Spam
    spam_probability = probability[1] * 100

    print("\n" + "="*60)
    print("PREDICTION RESULT")
    print("="*60)

    if prediction == 1:

        print("🚨 Email Status : SPAM EMAIL")
        print(f"Spam Confidence : {spam_probability:.2f}%")
        print(f"Ham Confidence  : {ham_probability:.2f}%")

    else:

        print("✅ Email Status : SAFE EMAIL (HAM)")
        print(f"Ham Confidence  : {ham_probability:.2f}%")
        print(f"Spam Confidence : {spam_probability:.2f}%")

    print("="*60)


✅ Logistic Regression Model Trained Successfully

MODEL PERFORMANCE
Accuracy : 100.00%

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       993
           1       1.00      1.00      1.00      1007

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000


Confusion Matrix
[[ 993    0]
 [   0 1007]]

MODEL FILES SAVED SUCCESSFULLY
✔ spam_model.pkl
✔ tfidf_vectorizer.pkl

📧 EMAIL SPAM DETECTION SYSTEM
Type 'exit' to stop the program.

Enter Email Text : Goodluckinc797@gmail.com

PREDICTION RESULT
✅ Email Status : SAFE EMAIL (HAM)
Ham Confidence  : 87.37%
Spam Confidence : 12.63%
